# Capítulo 1 — Análise Exploratória de Dados

Notebook com o **código** deste capítulo, para o Google Colab. Cada trecho vem precedido de uma breve explicação; o texto completo está no site do livro.

Rode a célula de **setup** abaixo primeiro (uma vez), depois as demais em ordem.

In [ ]:
# Setup (rode uma vez).
!pip install -q wquantiles
!curl -sO https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/formato.py   # baixa o ajudante de formatação do livro

## 1.1 — Elementos de Dados Estruturados

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)

Isso pareceria bobagem de nomenclatura se não tivesse consequência prática. O tipo de uma variável determina três coisas: qual gráfico faz sentido para ela, qual estatística pode ser calculada sobre ela, e como o software deve armazenar e validar seus valores. Errar o tipo — tratar ordinal como numérico, ou nominal como se tivesse ordem — não é um erro estético; é o tipo de erro que produz uma conclusão errada com aparência de rigor.

In [ ]:
estado = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/estados.csv")
estado.dtypes

Isso é o que acontece quando ninguém declara o tipo explicitamente: o software cai de volta no genérico. Funciona, mas descarta informação — informação que o próximo passo recupera.

In [ ]:
estado["Sigla"] = estado["Sigla"].astype("category")
print(estado["Sigla"].dtype)
print("categorias:", len(estado["Sigla"].cat.categories))

Categórico nominal não tem ordem. Categórico ordinal tem — e essa diferença não é decorativa, ela muda o que o software permite fazer.

In [ ]:
from pandas.api.types import CategoricalDtype

tamanho = CategoricalDtype(categories=["pequeno", "medio", "grande"], ordered=True)
s = pd.Series(["grande", "pequeno", "medio"], dtype=tamanho)

print("ordenado:", s.sort_values().tolist())
print("maior que 'pequeno'?", s.gt("pequeno").tolist())
print("máximo:", s.max())

Agora o contraste. O mesmo dado, declarado como categórico **nominal** (sem `ordered=True`):

In [ ]:
nominal = pd.Series(["grande", "pequeno", "medio"], dtype="category")  # sem ordered=True

try:
    nominal.gt("pequeno")
except TypeError as erro:
    print("TypeError:", erro)

## 1.2 — Dados Retangulares

In [ ]:
import pandas as pd

A forma retangular é tão comum na prática estatística que é fácil esquecer que é uma escolha, não uma lei da natureza: **linhas são registros** e **colunas são variáveis**. No Python, o objeto que implementa essa ideia é o `DataFrame` do pandas — uma matriz de dados com rótulos.

In [ ]:
estado = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/estados.csv")
print("linhas x colunas:", estado.shape)
estado.head()

`estado.info()` complementa o `.head()`: além do formato, mostra quantos valores não nulos cada coluna tem e quanto de memória a tabela ocupa.

In [ ]:
estado.info()

Toda linha de um DataFrame tem um rótulo — o **índice** — além de qualquer coluna de dados. Por padrão, o pandas usa a posição da linha: 0, 1, 2...

In [ ]:
print("índice padrão:", estado.index[:5].tolist())

# Um índice significativo torna a busca por rótulo natural
por_estado = estado.set_index("Sigla")
por_estado.loc["SP"]

## 1.3 — Estimativas de Localização

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import trim_mean
import wquantiles
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

$$\bar{x} = \frac{\sum_{i=1}^{n} x_i}{n}$$

In [ ]:
media = estado["Populacao"].mean()
print(f"Média: {num(media)}")

$$\bar{x}_{\text{aparada}} = \frac{\sum_{i=p+1}^{n-p} x_{(i)}}{n - 2p}$$

In [ ]:
media_aparada = trim_mean(estado["Populacao"], 0.1)
print(f"Média aparada (10%): {num(media_aparada)}")

- Com **n ímpar** — como aqui, n = 27 —, existe um único valor central: o 14º estado da fila ordenada. A mediana **é uma observação de verdade**, não uma conta.
- Com **n par**, não há um único valor central: a mediana é a **média dos dois valores do meio**, e é por isso que ela pode terminar em `,5` — não há meia pessoa, há uma média entre duas observações vizinhas.

In [ ]:
mediana = estado["Populacao"].median()
print(f"Mediana: {num(mediana)}")

# Qual estado é a mediana? Com n ímpar, ela é uma observação de verdade.
estado_mediano = estado.loc[estado["Populacao"] == mediana, "Estado"].iloc[0]
print(f"É a população de: {estado_mediano}")

Na população dos estados, porém, a moda não tem o que dizer:

In [ ]:
print(f"Valores únicos de população: {estado['Populacao'].nunique()} de {len(estado)}")

É com dado **categórico** que a moda vira a medida certa — porque, para ele, média e mediana nem chegam a existir. As causas de atraso de voo do aeroporto de Dallas/Fort Worth, o mesmo dataset da seção 1.6, não têm ordem nem soma entre si: não há "média" entre `Clima` e `Seguranca`. Mas há uma categoria que ocorre mais que as outras:

In [ ]:
dfw = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/dfw_airline.csv").rename(columns={
    "Carrier": "Companhia", "ATC": "ControleAereo", "Weather": "Clima",
    "Security": "Seguranca", "Inbound": "VooAnterior"})
print(f"Causa modal de atraso: {dfw.iloc[0].idxmax()}")

$$\bar{x}_w = \frac{\sum_{i=1}^{n} w_i x_i}{\sum_{i=1}^{n} w_i}$$

In [ ]:
media_pond = np.average(estado["Taxa.Homicidios"], weights=estado["Populacao"])
mediana_pond = wquantiles.median(estado["Taxa.Homicidios"], weights=estado["Populacao"])

print(f"Média simples     : {num(estado['Taxa.Homicidios'].mean(), 4)}")
print(f"Média ponderada   : {num(media_pond, 4)}")
print(f"Mediana ponderada : {num(mediana_pond, 4)}")

A média ponderada é a taxa de homicídios que uma pessoa média do Brasil enfrenta; a média simples é a taxa de um estado médio. São perguntas diferentes, com respostas diferentes — e a **direção** em que elas diferem não é uma lei da estatística, é um fato empírico sobre o país que está sendo medido.

In [ ]:
fig, ax = plt.subplots()

ax.hist(estado["Populacao"] / 1e6, bins=20, color="#b0c4d8", edgecolor="white")
ax.axvline(media / 1e6, color="#c0392b", linestyle="-", linewidth=2, label=f"Média: {num(media/1e6, 1)}M")
ax.axvline(media_aparada / 1e6, color="#e67e22", linestyle="--", linewidth=2, label=f"Média aparada: {num(media_aparada/1e6, 1)}M")
ax.axvline(mediana / 1e6, color="#27ae60", linestyle=":", linewidth=2.5, label=f"Mediana: {num(mediana/1e6, 1)}M")

ax.set_xlabel("População (milhões)")
ax.set_ylabel("Número de estados")
ax.legend()
plt.tight_layout()
plt.show()

## 1.4 — Estimativas de Variabilidade

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels import robust
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)
estado = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/estados.csv")

$$s = \sqrt{\frac{\sum_{i=1}^{n}(x_i - \bar{x})^2}{n-1}}$$

In [ ]:
desvio = estado["Populacao"].std()
print(f"Desvio-padrão: {num(desvio)}")

A **amplitude interquartil** (IQR) é a diferença entre o 75º e o 25º percentil — a largura da metade central dos dados, ignorando completamente o que acontece nas pontas:

In [ ]:
iqr = estado["Populacao"].quantile(0.75) - estado["Populacao"].quantile(0.25)
print(f"IQR: {num(iqr)}")

$$\text{MAD} = \frac{\text{mediana}\big(|x_i - \text{mediana}(x)|\big)}{0{,}6745}$$

In [ ]:
mad = robust.scale.mad(estado["Populacao"])
print(f"MAD: {num(mad)}")

# O mesmo cálculo, explicitamente — para mostrar de onde vem o 0,6745
mad_manual = abs(estado["Populacao"] - estado["Populacao"].median()).median() / 0.6744897501960817
print(f"MAD (manual): {num(mad_manual)}")

Os dois cálculos batem exatamente: a função `robust.scale.mad` da `statsmodels` não faz nada além dessa conta. O fator 0,6745 no denominador não é arbitrário — ele calibra o MAD para que, sob uma distribuição normal, o valor resultante seja diretamente comparável ao desvio-padrão. Sem essa calibração, MAD e desvio-padrão viveriam em escalas diferentes e comparar os dois números não teria sentido.

In [ ]:
mediana = estado["Populacao"].median()
fig, ax = plt.subplots()

ax.hist(estado["Populacao"] / 1e6, bins=20, color="#b0c4d8", edgecolor="white")
ax.axvline(mediana / 1e6, color="#2c3e50", linewidth=2, label=f"Mediana: {num(mediana/1e6, 1)}M")

for medida, valor, cor, estilo in [
    ("Desvio-padrão", desvio, "#c0392b", "-"),
    ("IQR", iqr, "#e67e22", "--"),
    ("MAD", mad, "#27ae60", ":"),
]:
    ax.axvline((mediana + valor) / 1e6, color=cor, linestyle=estilo, linewidth=2,
               label=f"{medida}: {num(valor/1e6, 1)}M")

ax.set_xlabel("População (milhões)")
ax.set_ylabel("Número de estados")
ax.legend()
plt.tight_layout()
plt.show()

Por ser uma razão entre duas grandezas na mesma unidade (o desvio $x - \bar{x}$ e o desvio-padrão $s$, ambos em habitantes, ou ambos em homicídios por 100 mil), o z-score é **adimensional** — um número puro, sem unidade. É essa ausência de unidade que permite comparar posições em escalas completamente diferentes: população se mede em dezenas de milhões, taxa de homicídios em dezenas por 100 mil habitantes, e olhar os dois números brutos lado a lado não diz nada sobre qual deles é mais extremo. Convertidos a z-score, os dois passam a viver na mesma escala.

In [ ]:
mu, sd = estado["Populacao"].mean(), estado["Populacao"].std(ddof=1)
sp = estado.loc[estado["Sigla"] == "SP", "Populacao"].iloc[0]
z_sp = (sp - mu) / sd
print(f"z-score da população de SP: {num(z_sp, 2)}")

mu_t, sd_t = estado["Taxa.Homicidios"].mean(), estado["Taxa.Homicidios"].std(ddof=1)
sp_t = estado.loc[estado["Sigla"] == "SP", "Taxa.Homicidios"].iloc[0]
print(f"z-score da taxa de homicídios de SP: {num((sp_t - mu_t) / sd_t, 2)}")

## 1.5 — Explorando a Distribuição dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)
estado = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/estados.csv")

O **percentil** P é o valor abaixo do qual está P% das observações. A mediana, que a seção 1.3 já usou, nada mais é do que o percentil 50 — o caso particular em que exatamente metade dos dados fica de cada lado. Falar em percentis é generalizar essa ideia para qualquer fração, não só a metade. O termo **quantil** é o mesmo conceito, expresso em proporção (0,05) em vez de porcentagem (5%); `quantile()` no pandas usa essa segunda convenção.

In [ ]:
percentis = [0.05, 0.25, 0.5, 0.75, 0.95]
tabela = pd.DataFrame(estado["Taxa.Homicidios"].quantile(percentis))
tabela.index = [f"{p:.0%}" for p in percentis]
tabela.transpose().map(lambda v: num(v, 3))

O **boxplot** é uma leitura visual direta dos percentis. A caixa vai do primeiro ao terceiro quartil — e a largura dessa caixa é exatamente o IQR que a seção 1.4 calculou como medida de dispersão. Não é uma coincidência nem uma analogia: é a mesma conta, olhada de dois ângulos diferentes. A linha dentro da caixa marca a mediana. Os bigodes se estendem até 1,5 vez o IQR além de cada quartil, e qualquer observação além disso é desenhada como um ponto individual, não absorvida pela escala do gráfico.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 5))
(estado["Populacao"] / 1e6).plot.box(ax=ax)
ax.set_ylabel("População (milhões)")
plt.tight_layout()
plt.show()

O boxplot resume a distribuição em cinco números, mas cinco números continuam sendo um resumo — a caixa não distingue uma distribuição uniforme de uma concentrada em dois grupos distantes, contanto que os quartis batam. O **gráfico de violino** ataca exatamente essa lacuna: em volta da mesma caixa central, ele desenha, espelhada dos dois lados, a curva de densidade estimada (a mesma ideia da próxima seção) — a largura em cada altura é proporcional à quantidade de observações ali.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 5))
ax.violinplot(estado["Populacao"] / 1e6, showmedians=True)
ax.set_ylabel("População (milhões)")
ax.set_xticks([])
plt.tight_layout()
plt.show()

A **tabela de frequência** divide o intervalo de valores em faixas de largura igual e conta quantas observações caem em cada uma. `pd.cut` faz exatamente isso.

In [ ]:
faixas = pd.cut(estado["Populacao"], 10)
faixas.value_counts().sort_index()

O **histograma** é essa mesma tabela desenhada: uma barra por faixa, com altura proporcional à contagem. As barras se encostam porque o eixo é contínuo — não há espaço entre "faixa 1" e "faixa 2" do jeito que há entre categorias distintas.

In [ ]:
fig, ax = plt.subplots()
(estado["Populacao"] / 1e6).plot.hist(ax=ax, bins=10, edgecolor="white")
ax.set_xlabel("População (milhões)")
ax.set_ylabel("Número de estados")
plt.tight_layout()
plt.show()

A **curva de densidade** é uma versão suavizada do histograma: em vez de barras discretas, uma curva contínua que estima a forma subjacente da distribuição. A diferença crucial é a escala do eixo Y. Enquanto o histograma tradicional mostra contagem, a densidade é normalizada para que a área sob a curva valha exatamente 1 — é o preço de tratá-la como uma distribuição de probabilidade, não como uma lista de frequências.

In [ ]:
fig, ax = plt.subplots()
estado["Taxa.Homicidios"].plot.hist(ax=ax, density=True, xlim=[0, 40],
                                bins=range(0, 40, 2), edgecolor="white")
estado["Taxa.Homicidios"].plot.density(ax=ax, linewidth=2)
ax.set_xlabel("Taxa de homicídios (por 100.000)")
plt.tight_layout()
plt.show()

As seções 1.3 e 1.4 reduziram a população dos estados a dois números: um valor típico e uma medida de quanto os dados se espalham em torno dele. Considere agora três turmas fictícias de 25 alunos, com boletins **idênticos**: média 65, desvio-padrão 12. Se essas duas estatísticas fossem tudo que existisse para descrever uma turma, seria razoável supor que as três se parecem. Elas não se parecem: uma tem a maioria das notas abaixo da média com um rastro de notas altas puxando para cima; outra é o espelho exato dessa primeira; a terceira não tem rastro para nenhum lado. Média e variância dizem **onde** os dados estão centrados e **quão espalhados** eles estão — nenhuma das duas diz uma palavra sobre a **forma**.

In [ ]:
N, MEDIA, DESVIO = 25, 65.0, 12.0

def padroniza(x, media=MEDIA, desvio=DESVIO):
    """Crava média e desvio exatos, via z-score e reescala.

    A assimetria é invariante a transformação linear — ela sobrevive intacta.
    É isso que permite três conjuntos com a MESMA média e a MESMA variância,
    mas formas diferentes.
    """
    x = np.asarray(x, float)
    return (x - x.mean()) / x.std(ddof=1) * desvio + media

# Um gerador por turma: a ordem de consumo do RNG não pode alterar um conjunto
# em silêncio quando outro mudar.
turma_a = padroniza(np.random.default_rng(42).lognormal(0, 1.0, N))

# Espelho em torno da média: preserva média e variância, e inverte o SINAL da
# assimetria — exatamente, não aproximadamente.
turma_b = 2 * MEDIA - turma_a

# Simétrica POR CONSTRUÇÃO, não por amostragem: 12 desvios, seus 12 espelhos,
# e o centro. A assimetria é exatamente zero, não "próxima de zero".
desvios = np.abs(np.random.default_rng(7).normal(0, 1, (N - 1) // 2))
turma_c = padroniza(np.concatenate([-desvios, [0.0], desvios]))

turmas = {"A — à direita": turma_a, "B — à esquerda": turma_b, "C — simétrica": turma_c}

# As garantias são verificadas, não afirmadas.
assert np.allclose([t.mean() for t in turmas.values()], MEDIA)
assert np.allclose([t.var(ddof=1) for t in turmas.values()], DESVIO ** 2)
assert np.isclose(stats.skew(turma_a, bias=False), -stats.skew(turma_b, bias=False))
assert np.isclose(stats.skew(turma_c, bias=False), 0, atol=1e-12)
assert all(t.min() >= 0 and t.max() <= 100 for t in turmas.values())

As quatro estatísticas a seguir tornam a diferença impossível de ignorar:

In [ ]:
resumo = pd.DataFrame(
    {
        "Média": [num(t.mean(), 1) for t in turmas.values()],
        "Mediana": [num(np.median(t), 1) for t in turmas.values()],
        "Variância": [num(t.var(ddof=1), 1) for t in turmas.values()],
        "Assimetria": [num(stats.skew(t, bias=False), 2) for t in turmas.values()],
    },
    index=list(turmas),
)
resumo

O histograma deixa a mesma história visível:

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(11, 3.3), sharey=True)

for ax, (nome, notas) in zip(eixos, turmas.items()):
    ax.hist(notas, bins=np.arange(30, 101, 7), color="#b0c4d8", edgecolor="white")
    ax.axvline(notas.mean(), color="#c0392b", linewidth=2)
    ax.axvline(np.median(notas), color="#27ae60", linestyle="--", linewidth=2)
    ax.set_title(nome, fontsize=10)
    ax.set_xlabel("Nota")

eixos[0].set_ylabel("Alunos")
plt.tight_layout()
plt.show()

O histograma mostra a forma inteira, mas a assimetria aparece igualmente bem no **boxplot** — que resume cada turma em cinco números. Empilhando os três lado a lado, a diferença salta aos olhos mesmo sem contar nenhuma nota:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))

caixas = ax.boxplot(
    [turma_c, turma_b, turma_a],          # de baixo para cima: simétrica, esquerda, direita
    tick_labels=["C — simétrica", "B — à esquerda", "A — à direita"],
    vert=False,
    patch_artist=True,
    medianprops={"color": "#27ae60", "linewidth": 2},
    widths=0.6,
)
for caixa in caixas["boxes"]:
    caixa.set(facecolor="#b0c4d8", edgecolor="#2c3e50")

ax.set_xlabel("Nota")
plt.tight_layout()
plt.show()

Voltando aos dados reais das 27 unidades federativas:

In [ ]:
print(f"População das UFs  : {num(stats.skew(estado['Populacao'], bias=False), 2)}")
print(f"Taxa de homicídios : {num(stats.skew(estado['Taxa.Homicidios'], bias=False), 2)}")

A assimetria mede para qual **lado** a distribuição pende. Ela não mede outra coisa, igualmente relevante: o quão pesadas são as caudas, não importa o lado — ou seja, quão comuns são os valores extremos, contra o que se esperaria de uma normal. Essa é a pergunta que a **curtose** responde.

In [ ]:
print(f"Curtose da população das UFs : {num(stats.kurtosis(estado['Populacao']), 2)}")
print(f"Curtose da taxa de homicídios: {num(stats.kurtosis(estado['Taxa.Homicidios']), 2)}")

## 1.6 — Explorando Dados Binários e Categóricos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

Para dado numérico, o primeiro resumo é uma medida de localização — média, média aparada, mediana. Nenhuma delas se aplica aqui: somar "Companhia + Clima" e dividir por dois não produz nada interpretável, porque não há distância nem ordem entre as categorias. O que resta, e é suficiente, é a **proporção** de cada categoria sobre o total.

In [ ]:
dfw = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/dfw_airline.csv").rename(columns={
    "Carrier": "Companhia",
    "ATC": "ControleAereo",
    "Weather": "Clima",
    "Security": "Seguranca",
    "Inbound": "VooAnterior",
})
proporcoes = 100 * dfw / dfw.values.sum()
proporcoes.round(2).map(lambda v: num(v, 2))

`ControleAereo` e `VooAnterior` somam sozinhos mais de 70% dos atrasos; `Seguranca` responde por pouco mais de um décimo de 1%. É toda a "distribuição" que um dado categórico tem para oferecer — não uma forma contínua, mas um conjunto de fatias.

In [ ]:
fig, ax = plt.subplots()
dfw.transpose().plot.bar(ax=ax, legend=False, color="#4a90a4", edgecolor="white")
ax.set_xlabel("Causa do atraso")
ax.set_ylabel("Número de atrasos")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

O gráfico abaixo mostra a taxa de homicídios dos nove estados do Nordeste (Atlas da Violência, 2024), com os estados em ordem alfabética da sigla.

In [ ]:
estado = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/estados.csv")
nordeste = ["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"]
ne = estado[estado["Sigla"].isin(nordeste)].sort_values("Sigla")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(ne["Sigla"], ne["Taxa.Homicidios"], color="#b0c4d8", edgecolor="white")
ax.set_ylim(20, 38)
ax.set_xlabel("Estado")
ax.set_ylabel("Taxa de homicídios (por 100 mil)")
plt.tight_layout()
plt.show()

Veja o que acontece quando a régua começa onde deveria, no zero:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(ne["Sigla"], ne["Taxa.Homicidios"], color="#b0c4d8", edgecolor="white")
ax.set_ylim(0, 38)
ax.set_xlabel("Estado")
ax.set_ylabel("Taxa de homicídios (por 100 mil)")
plt.tight_layout()
plt.show()

**Regra prática:** um gráfico de barras começa no eixo zero, sempre. A barra codifica a magnitude pelo seu comprimento, e comprimento medido a partir de um ponto qualquer não é magnitude nenhuma. (A regra é específica de barras: um gráfico de **linha**, mostrando evolução no tempo, às vezes pode truncar o eixo sem enganar — ali a informação está na **inclinação** da linha, não no comprimento de uma barra medida contra o zero.)

In [ ]:
moda = proporcoes.transpose().iloc[:, 0].idxmax()
print(f"Moda (causa mais frequente): {moda}")

Suponha que uma companhia aérea estime o custo médio de compensação por passageiro, e que esse custo dependa da causa do atraso:

In [ ]:
# Uma companhia estima o custo médio de compensação por passageiro,
# conforme a causa do atraso.
custo = {"Companhia": 180, "ControleAereo": 40, "Clima": 0, "Seguranca": 25, "VooAnterior": 120}

p = (dfw / dfw.values.sum()).transpose().iloc[:, 0]   # probabilidade de cada causa
ve = sum(p[causa] * valor for causa, valor in custo.items())

print(f"Valor esperado do custo por atraso: R$ {num(ve, 2)}")

## 1.7 — Correlação

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

Vamos usar retornos diários de ações e ETFs (fundos negociados em bolsa) do S&P 500, entre 1993 e 2015.

In [ ]:
SETORES = {
    "consumer_discretionary": "consumo_discricionario",
    "consumer_staples": "consumo_essencial",
    "energy": "energia",
    "etf": "etf",
    "financials": "financeiro",
    "health_care": "saude",
    "industrials": "industrial",
    "information_technology": "tecnologia_da_informacao",
    "materials": "materiais",
    "telecommunications_services": "telecomunicacoes",
    "utilities": "utilidade_publica",
}
setores = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/sp500_sectors.csv").rename(
    columns={"sector": "Setor", "symbol": "Simbolo"}
)
setores["Setor"] = setores["Setor"].replace(SETORES)
precos  = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/sp500_data.csv.gz", index_col=0)

simbolos_telecom = setores[setores["Setor"] == "telecomunicacoes"]["Simbolo"]
telecom = precos.loc[precos.index >= "2012-07-01", simbolos_telecom]

print("dias x empresas:", telecom.shape)
telecom.head()

Com mais de duas variáveis, faz sentido calcular a correlação de **todo par**, não de um par isolado. O resultado é a **matriz de correlação**: simétrica (a correlação de T com VZ é a mesma de VZ com T) e com 1 na diagonal (toda variável tem correlação perfeita consigo mesma).

In [ ]:
telecom.corr().round(3).map(lambda v: num(v, 3))

A maior correlação do grupo é entre T e VZ: …. Não é coincidência: AT&T e Verizon são as duas maiores operadoras de telecomunicações dos Estados Unidos, e seus retornos diários respondem aos mesmos choques de mercado — taxa de juros, regulação do setor, expectativa de crescimento da economia. Empresas menores e mais heterogêneas do grupo, como LVLT, têm correlação bem mais baixa com as demais (0,242 a 0,287): compartilham menos desses choques comuns.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(telecom["T"], telecom["VZ"], alpha=0.5, s=20, color="#2c7fb8")
ax.axhline(0, color="grey", linewidth=0.8)
ax.axvline(0, color="grey", linewidth=0.8)
ax.set_xlabel("Retorno diário — AT&T (T)")
ax.set_ylabel("Retorno diário — Verizon (VZ)")
plt.tight_layout()
plt.show()

Uma matriz de 5 × 5 ainda se lê em números. Uma de 17 × 17 não. Nesses casos, transformar a matriz em uma imagem colorida — um **heatmap** — é o único jeito prático de enxergar o padrão.

In [ ]:
etfs = precos.loc[precos.index > "2012-07-01",
                  setores[setores["Setor"] == "etf"]["Simbolo"]]

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(etfs.corr(), vmin=-1, vmax=1,
            cmap=sns.diverging_palette(20, 220, as_cmap=True), ax=ax)
plt.tight_layout()
plt.show()

A terceira armadilha é a mais contraintuitiva das três, e vale prová-la em vez de só declará-la.

In [ ]:
x = np.linspace(-1, 1, 200)
y = x ** 2

print(f"Correlação entre x e x²: {np.corrcoef(x, y)[0, 1]:.3e}")

## 1.8 — Explorando Duas ou Mais Variáveis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

Vamos usar avaliações fiscais de imóveis do condado de King, em Washington: área construída e valor venal.

In [ ]:
kc = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/kc_tax.csv.gz").rename(columns={
    "TaxAssessedValue": "ValorVenal",
    "SqFtTotLiving": "AreaConstruida",
    "ZipCode": "CEP",
})
print(f"registros brutos: {num(len(kc), 0)}")

kc0 = kc.loc[(kc.ValorVenal < 750000) &
             (kc.AreaConstruida > 100) &
             (kc.AreaConstruida < 3500), :]
print(f"após filtrar extremos: {num(len(kc0), 0)}")

Duas ferramentas resolvem isso sem descartar pontos: o **hexbin**, que divide o plano em células hexagonais e colore cada uma pela contagem, e o **contorno de densidade** (KDE), que estima uma superfície contínua de densidade e a desenha como curvas de nível.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
kc0.plot.hexbin(x="AreaConstruida", y="ValorVenal",
                gridsize=30, sharex=False, ax=ax)
ax.set_xlabel("Área construída (pés²)")
ax.set_ylabel("Valor venal (US$)")
plt.tight_layout()
plt.show()

Esta é a primeira seção do livro com um componente aleatório, e por isso a primeira em que a semente importa de fato. Sem `random_state=42`, cada `quarto render` sortearia 10.000 imóveis diferentes, e o contorno mudaria de forma sutil a cada execução: o gráfico publicado nunca seria o mesmo duas vezes, o cache do `freeze` perderia sentido — ele reexecuta um chunk quando o texto muda, não quando o sorteio decide mudar sozinho —, e cada `push` acumularia um diff de ruído no site. É por isso que todo chunk deste livro com gerador aleatório fixa a semente, e é aqui que essa regra encontra seu primeiro caso real.

In [ ]:
amostra = kc0.sample(10000, random_state=42)

fig, ax = plt.subplots(figsize=(6, 5))
sns.kdeplot(data=amostra, x="AreaConstruida", y="ValorVenal", ax=ax)
ax.set_xlabel("Área construída (pés²)")
ax.set_ylabel("Valor venal (US$)")
plt.tight_layout()
plt.show()

Vamos usar empréstimos do LendingClub, classificados por nota de risco (`Nota`, de A — a melhor — a G) e por situação atual (`Situacao`).

In [ ]:
SITUACAO = {
    "Fully Paid": "Quitado",
    "Current": "Em dia",
    "Late": "Atrasado",
    "Charged Off": "Inadimplente",
}
lc = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/lc_loans.csv").rename(columns={"status": "Situacao", "grade": "Nota"})
lc["Situacao"] = lc["Situacao"].replace(SITUACAO)

contagem = lc.pivot_table(index="Nota", columns="Situacao",
                          aggfunc=lambda x: len(x), margins=True)
contagem

Olhe a coluna `Inadimplente`. A nota C tem 6.023 empréstimos inadimplentes — mais que qualquer outra nota, mais que E, F e G somadas. É tentador ler isso como "C é a nota mais arriscada da carteira". Mas a contagem mistura duas coisas: o quão arriscada é a nota, e o quão grande é a nota. B e C sozinhas concentram mais da metade da carteira (132.370 e 120.875 empréstimos), então é claro que têm mais de tudo, inclusive mais inadimplência. Comparar contagens brutas entre grupos de tamanhos tão diferentes é um erro clássico, e é exatamente o que a tabela de contagem convida a fazer.

In [ ]:
prop = contagem.copy().loc["A":"G", :].astype(float)
situacoes = [c for c in prop.columns if c != "All"]
prop.loc[:, situacoes] = prop.loc[:, situacoes].div(prop["All"], axis=0)
prop["All"] = prop["All"] / sum(prop["All"])
prop.round(3).map(lambda v: num(v, 3))

Vamos usar o percentual diário de voos atrasados por causa atribuível à própria companhia aérea (`pct_atraso_companhia`), por companhia (`Companhia`).

In [ ]:
voos = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/airline_stats.csv").rename(columns={
    "pct_carrier_delay": "pct_atraso_companhia",
    "pct_atc_delay": "pct_atraso_controle",
    "pct_weather_delay": "pct_atraso_clima",
    "airline": "Companhia",
})

fig, ax = plt.subplots(figsize=(7, 5))
voos.boxplot(by="Companhia", column="pct_atraso_companhia", ax=ax)
ax.set_xlabel("")
ax.set_ylabel("% diário de voos atrasados")
ax.set_ylim(0, 50)
plt.suptitle("")
plt.title("")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

São 33.468 dias-companhia, distribuídos entre 6 companhias aéreas. As medianas (a linha dentro de cada caixa) ordenam as companhias de forma nítida: Alaska tem a mediana mais baixa, 3,23% — a melhor do grupo —; American tem a mais alta, 8,43% — a pior. No meio, Delta (5,55%), United (6,45%), Southwest (6,96%) e Jet Blue (7,66%) ocupam posições intermediárias, nessa ordem.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.violinplot(data=voos, x="Companhia", y="pct_atraso_companhia",
               ax=ax, inner="quartile", color="#b0c4d8")
ax.set_xlabel("")
ax.set_ylabel("% diário de voos atrasados")
ax.set_ylim(0, 50)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

Tudo até aqui comparou exatamente duas variáveis. Perguntas reais costumam ter uma terceira à espreita: será que a relação entre área e valor do imóvel é a mesma em qualquer lugar, ou ela muda dependendo do bairro? A forma de responder é **condicionar**: fixar um valor da terceira variável de cada vez e repetir o gráfico de par para cada um. Visualmente, isso é uma grade de painéis pequenos — um `FacetGrid` — em vez de um único gráfico agregado.

In [ ]:
kc_ceps = kc0.loc[kc0.CEP.isin([98188, 98105, 98108, 98126]), :]

def hexbin(x, y, color, **kwargs):
    cmap = sns.light_palette(color, as_cmap=True)
    plt.hexbin(x, y, gridsize=25, cmap=cmap, **kwargs)

g = sns.FacetGrid(kc_ceps, col="CEP", col_wrap=2, height=3.2)
g.map(hexbin, "AreaConstruida", "ValorVenal", extent=[0, 3500, 0, 700000])
g.set_axis_labels("Área construída (pés²)", "Valor venal (US$)")
plt.tight_layout()
plt.show()